# Notebook 3 — Build a FODO ring

In this first exercise, use the tools introduced in the two introductory notebooks to **build an electron storage ring from identical FODO cells**. Work in this notebook; use the introductory notebooks as references without modifying them.

We will later study the optics and radiation equilibrium of this ring and compare it with a light-source lattice. For now, concentrate on defining the elements, assembling the ring, and viewing its layout.

## Lattice specifications

| Parameter | Value |
|---|---:|
| Reference particle | Electron |
| Reference total energy | 1.7 GeV |
| Ring circumference | 240 m |
| Number of identical FODO cells | 16 |
| Cell length | 15 m |
| Dipoles per cell | 2 |
| Dipole type | Sector bend (`xt.Bend`), with no quadrupole component |
| Dipole length | 4.0 m |
| Quadrupoles per cell | One horizontally focusing QF and one horizontally defocusing QD |
| Quadrupole length | 0.30 m |

## Your tasks

1. **Create an Xtrack environment.** Set a 1.7 GeV reference electron. Define variables for the magnet lengths, dipole angle, and two independent quadrupole strengths, `kqf` and `kqd`. Use deferred expressions to connect the elements to these variables. For now, set `kqf = 0.65` m$^{-2}$ and `kqd = -0.65` m$^{-2}$; these are starting values, not matched strengths.
2. **Define the magnets and assemble one FODO cell.** Position the magnets with `env.place(..., at=...)` and let Xtrack create the drifts.
3. **Assemble the ring from 16 copies of the cell.** Give the magnet occurrences unique names with `replace_all_repeated_elements()`, while retaining the common family knobs.
4. **Plot the ring layout.** Compute the survey with `ring.survey()` and display it with `sv.plot()`.

*Optics matching, sextupoles, RF, and radiation are left for subsequent steps.*

## Worked solution

This is one possible layout. We place a dipole halfway between each pair of neighbouring quadrupoles, leaving four equal drifts per cell. The two quadrupole families have independent strength knobs.

### Create the environment and define the parameters

In [ ]:
%matplotlib inline
import numpy as np
import xtrack as xt

In [ ]:
env = xt.Environment()
env.set_particle_ref('electron', energy0=1.7e9)

n_cells = 16
n_bends = 2 * n_cells

env['l_cell'] = 15.0
env['l_bend'] = 4.0
env['l_quad'] = 0.30
env['angle_bend'] = 2 * np.pi / n_bends

env['kqf'] = 0.65
env['kqd'] = -0.65

### Define the magnets and build the cell

In [ ]:
# Define elements
env.new('qf', xt.Quadrupole, length='l_quad', k1='kqf')
env.new('qd', xt.Quadrupole, length='l_quad', k1='kqd')
env.new('bend', xt.Bend, length='l_bend', angle='angle_bend', k1=0)

In [ ]:
# Define cell
cell = env.new_line(length=env['l_cell'], components=[
    env.place('qf',   at='l_quad/2'),
    env.place('bend', at='l_quad/2 + l_cell/4'),
    env.place('qd',   at='l_quad/2 + l_cell/2'),
    env.place('bend', at='l_quad/2 + 3*l_cell/4'),
])

### Assemble the ring

Repeat the cell 16 times, then give each magnet occurrence a unique name. The magnets remain linked to the same family variables.

In [ ]:
ring = n_cells * cell
ring.replace_all_repeated_elements()

### Survey

In [ ]:
sv = ring.survey()

In [ ]:
sv.plot()